# 11 — Prompt sensitivity (RQ3): built-in vs custom V2 prompt

**Research question 3.** Is there a difference between the `llm-feature-gen` library's
built-in default prompt (used for V1 categorical extraction) and the custom V2 prompt
(count + evidence schema)?

**What this notebook does.** Runs a focused CV comparison between three feature
representations:
1. **Surface stats** — length and lexical diversity, the confound baseline.
2. **V1 built-in prompt** — 10 categorical features produced by llm-feature-gen's
   discovery + generation pipeline (cached in `OutputsQwen/`).
3. **V2 custom prompt** — frozen primary (13 stable V2 rates + 7 surface stats,
   `pipe_v2_stable_surface.joblib`).

The same 10×5 repeated-CV protocol and seed as notebooks 08/09 are used so that
out-of-fold probabilities are comparable across arms and paired bootstrap is valid.

**Test set is not loaded.** This notebook answers RQ3 on the training set. The
one-shot sealed-test evaluation is notebook 10.

## Block 0 — Configuration

Prompt hash and domain-blind assertions are re-checked here even though this notebook
does not run a new extraction — the V2 features come from `v2_raw.jsonl` (already on
disk), and the hash tells us the cached file was produced under the frozen prompt, not
an earlier draft.

In [1]:
import json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold

from helpers import (
    MODEL_A as MODEL, SEED, EXPECTED_PROMPT_SHA,
    V2_PROMPT, PROMPT_SHA, assert_prompt_hash, assert_domain_blind,
    evaluate as _pkg_evaluate, paired_bootstrap,
    N_REPEATS_DEFAULT as N_REPEATS, N_SPLITS_DEFAULT as N_SPLITS,
)

warnings.filterwarnings("ignore")

DATA       = Path("fileDataset")
OUT        = DATA / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
FROZEN_DIR = OUT / "frozen_pipeline_v2"
RAW_A      = OUT / "v2_raw.jsonl"
if not DATA.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA}. Put fileDataset/ next to this notebook.")

assert_prompt_hash(EXPECTED_PROMPT_SHA)
assert_domain_blind()
print(f"prompt sha256[:16] = {PROMPT_SHA}  ({len(V2_PROMPT)} chars)")
print(f"CV protocol: {N_REPEATS} repeats x {N_SPLITS}-fold StratifiedKFold, seed={SEED}")
print("TEST SET: not loaded -- training-only notebook")


prompt sha256[:16] = f52aae5d939d620e  (2428 chars)
CV protocol: 10 repeats x 5-fold StratifiedKFold, seed=42
TEST SET: not loaded -- training-only notebook


## Block 1 — Load labelled training data

241 labelled `train/` documents only. The 86 unlabelled `overview/` documents are
loaded (as `load_labelled_and_unlabelled` requires) so that V2 feature construction
works correctly, but they never enter the CV loop. `test/` is never read.

In [2]:
# 241 labelled train/ docs. The 86 unlabelled overview/ docs are loaded too (V2
# feature construction needs them) but never enter the CV loop. test/ is never read.
def read_split(folder, label):
    return [{"file": f.name, "label": label,
             "text": f.read_text(encoding="utf-8", errors="replace").strip()}
            for f in sorted(folder.glob("*.txt"))]

_WORD = re.compile(r"\w+", re.UNICODE)
def surface(t):
    nw = max(1, len(t.split()))
    nt = len(set(_WORD.findall(t.lower())))
    ns = max(1, len(re.findall(r"[.!?]+", t)))
    return {"n_char": len(t), "n_word": nw, "n_type": nt, "ttr": nt / nw,
            "n_sent": ns, "mlu": nw / ns, "n_comma": t.count(",")}

rows = (read_split(DATA / "train" / "negative", 0)
        + read_split(DATA / "train" / "positive", 1)
        + read_split(DATA / "overview", None))
docs = pd.DataFrame(rows)
docs = pd.concat([docs, docs.text.apply(lambda t: pd.Series(surface(t)))], axis=1)
SURFACE = ["n_word", "n_type", "ttr", "n_sent", "mlu", "n_comma", "n_char"]

n_lab, n_unlab = int(docs.label.notna().sum()), int(docs.label.isna().sum())
assert (n_lab, n_unlab) == (241, 86), (n_lab, n_unlab)

lab = docs[docs.label.notna()].copy()
lab["label"] = lab.label.astype(int)
y = lab.label.values
print(f"labelled docs: {len(lab)}  (pos {y.sum()}, neg {(y==0).sum()})")
print("test/ was not read")


labelled docs: 241  (pos 70, neg 171)
test/ was not read


## Block 2 — V1 features: built-in llm-feature-gen prompt (cached)

V1 used llm-feature-gen's two-step pipeline: `text_discovery_prompt` on the
unlabelled `overview/` documents to propose features, then `text_generation_prompt`
on the labelled `train/` documents to assign values. The result is 10 categorical
string features per document, OHE-encoded downstream. The cached extraction lives
in `OutputsQwen/train_all_feature_values.csv` — no LLM call is needed here.

In [3]:
v1_path = next((
    p for p in [
        Path("../OutputsQwen/train_all_feature_values.csv"),
        Path("OutputsQwen/train_all_feature_values.csv"),
        Path("../outputs/v1_train.csv"),
    ] if p.exists()
), None)
assert v1_path is not None, (
    "V1 feature CSV not found. Expected at OutputsQwen/train_all_feature_values.csv "
    "(relative to the repo root or the notebooks_preliminary/ folder)."
)

v1 = pd.read_csv(v1_path).rename(columns={"File": "file"})
V1_COLS = [c for c in v1.columns if c not in ("file", "Class", "raw_llm_output")]

m1 = lab[["file"]].merge(v1[["file"] + V1_COLS], on="file", how="left").fillna("missing")
assert len(m1) == len(lab), f"V1 merge: expected {len(lab)} rows, got {len(m1)}"
coverage = m1[V1_COLS].ne("missing").all(axis=1).mean()

print(f"V1 features: {len(V1_COLS)} categorical columns, {coverage:.0%} complete coverage")
print(f"  {V1_COLS}")

V1 features: 10 categorical columns, 100% complete coverage
  ['narrative_coherence', 'uncertainty_markers', 'action_verb_tense_consistency', 'entity_specificity_level', 'self_referential_metacommentary', 'spatial_organization_pattern', 'emotional_interpretation_inference', 'lexical_variation_and_repetition', 'quantification_precision', 'discourse_marker_usage']


## Block 3 — V2 features: custom V2 prompt (frozen primary)

Load the cached `v2_raw.jsonl` produced by notebook 08 (MODEL_A, `qwen3.5:122b`).
The frozen primary feature set (`V2_STABLE + surface`, 20 columns) is read from
`frozen_spec.json` — the same source notebook 10 uses — so this notebook never needs
to re-derive or hand-pick the column list.

In [4]:
spec = json.loads((FROZEN_DIR / "frozen_spec.json").read_text())
FROZEN_PRIMARY_FEATURES = spec["feature_sets"]["frozen_primary"]

# load cached v2_raw.jsonl (notebook 08, MODEL_A) and build the feature matrix -- inline
def load_raw(path):
    out = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            try:
                rec = json.loads(line); out[rec["file"]] = rec
            except Exception:
                pass
    return out

LIST_FIELDS = {"n_entities": "named_entities", "n_specific_verb": "specific_action_verbs",
               "n_generic_verb": "generic_verbs", "n_locative": "locative_expressions",
               "n_hedge": "hedge_spans", "n_deictic": "deictic_spans",
               "n_metacomment": "metacomment_spans", "n_diminutive": "diminutive_or_affective_forms",
               "n_quantity": "quantity_expressions"}
INT_FIELDS = {"n_proposition": "complete_propositions", "n_selfcorrect": "self_corrections"}

def to_row(obj):
    r = {}
    for out_, key in LIST_FIELDS.items():
        v = obj.get(key) or []
        r[out_] = len(v) if isinstance(v, list) else 0
    for out_, key in INT_FIELDS.items():
        v = obj.get(key, 0)
        r[out_] = int(v) if isinstance(v, (int, float)) else 0
    reg = obj.get("regions_referenced") or []
    r["n_region"] = len(set(reg)) if isinstance(reg, list) else 0
    rep = obj.get("repeated_content_lemmas") or []
    r["n_repeated_lemma"] = sum(1 for x in rep if isinstance(x, dict) and x.get("lemma"))
    r["repeat_mass"] = sum(int(x.get("count", 0)) for x in rep
                           if isinstance(x, dict) and str(x.get("count", "")).isdigit())
    return r

assert RAW_A.exists(), f"{RAW_A} not found -- run notebook 08 first"
raw_A = load_raw(RAW_A)
ok_A  = {f: v["response"] for f, v in raw_A.items() if v["response"] is not None}

feat = pd.DataFrame([{"file": f, **to_row(o)} for f, o in ok_A.items()])
dfA = docs.merge(feat, on="file", how="inner")
for c in [c for c in feat.columns if c != "file"]:
    dfA[c + "_r100"] = 100 * dfA[c] / dfA["n_word"].clip(lower=1)
dfA["specific_verb_ratio"] = dfA.n_specific_verb / (dfA.n_specific_verb + dfA.n_generic_verb).clip(lower=1)
dfA["region_breadth"] = dfA.n_region / 3.0

labA = dfA[dfA.label.notna()].copy()
labA["label"] = labA.label.astype(int)
labA = labA.set_index("file").loc[lab.file].reset_index()
assert list(labA.file)  == list(lab.file),  "document order mismatch between labA and lab"
assert list(labA.label) == list(lab.label), "label mismatch between labA and lab"
assert all(c in labA.columns for c in FROZEN_PRIMARY_FEATURES), "missing frozen_primary columns"

print(f"V2 extraction: {len(ok_A)}/{len(raw_A)} docs ({len(ok_A)/len(raw_A):.0%} success)")
print(f"Frozen primary: {len(FROZEN_PRIMARY_FEATURES)} cols  "
      f"(V2_STABLE {len(spec['feature_sets']['V2_STABLE'])} + surface {len(spec['feature_sets']['SURFACE'])})")
print(f"  {FROZEN_PRIMARY_FEATURES}")


V2 extraction: 327/327 docs (100% success)
Frozen primary: 20 cols  (V2_STABLE 13 + surface 7)
  ['n_specific_verb_r100', 'n_locative_r100', 'n_hedge_r100', 'n_deictic_r100', 'n_metacomment_r100', 'n_diminutive_r100', 'n_quantity_r100', 'n_proposition_r100', 'n_selfcorrect_r100', 'n_region_r100', 'n_repeated_lemma_r100', 'repeat_mass_r100', 'region_breadth', 'n_word', 'n_type', 'ttr', 'n_sent', 'mlu', 'n_comma', 'n_char']


## Block 4 — CV comparison

Three arms, same 10×5 repeated-CV protocol, same `SEED` → identical fold assignments
across arms, which is required for the paired bootstrap in Block 5 to be valid.

* **Surface stats** — length confound baseline, ElasticNet (nested CV for `C`/`l1_ratio`).
* **V1 built-in prompt** — OHE (leak-free: encoder inside the pipeline) + plain L2
  logistic regression (same as notebooks 04/08 for comparability).
* **V2 custom prompt** — frozen primary feature set, ElasticNet (same classifier as
  the frozen primary pipeline in notebooks 09/10).

V1 uses L2 instead of ElasticNet because its 10-dimensional OHE space is already
sparse and small; nested ElasticNet CV would add variance without benefit.

In [5]:
def enet():
    return make_pipeline(StandardScaler(), LogisticRegressionCV(
        penalty="elasticnet", solver="saga", l1_ratios=[0.3, 0.6, 0.9], Cs=8,
        max_iter=5000, class_weight="balanced", scoring="roc_auc",
        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1))

# run() fits + scores one arm silently -- the table printed below is the only
# place these numbers get shown, so nothing here duplicates it.
PROBA, res = {}, []
def run(name, model, X, key):
    r = _pkg_evaluate(name, X, y, seed=SEED, model=model)
    PROBA[key] = r["_proba"]
    res.append({k: v for k, v in r.items() if k != "_proba"})

print(f"fitting 3 arms — 10x{N_SPLITS} repeated CV (seed={SEED})...")
run("Surface stats only", enet(), lab[SURFACE].values, "surf")
run("V1 built-in prompt (OHE + L2 logreg)",
    make_pipeline(OneHotEncoder(handle_unknown="ignore"),
                  LogisticRegression(max_iter=2000, class_weight="balanced")),
    m1[V1_COLS], "v1")
run("V2 custom prompt -- frozen primary (V2_STABLE + surface)",
    enet(), labA[FROZEN_PRIMARY_FEATURES].values, "v2")

table = pd.DataFrame(res)
print()
print(table.round(3).to_string(index=False))
table.to_csv(OUT / "v11_prompt_sensitivity.csv", index=False)
print(f"\nsaved -> {OUT / 'v11_prompt_sensitivity.csv'}")

fitting 3 arms — 10x5 repeated CV (seed=42)...

                                                   model  n_feat   AUC  CI_low  CI_high  macroF1  balAcc
                                      Surface stats only       7 0.743   0.679    0.805    0.642   0.662
                    V1 built-in prompt (OHE + L2 logreg)      10 0.693   0.617    0.764    0.614   0.625
V2 custom prompt -- frozen primary (V2_STABLE + surface)      20 0.806   0.738    0.867    0.744   0.756

saved -> fileDataset/outputs/v11_prompt_sensitivity.csv


## Block 5 — Paired bootstrap (RQ3 decisive contrasts)

Three contrasts: V2 vs V1 (the direct RQ3 answer), V2 vs surface, V1 vs surface.
Because all three arms used the same SEED the OOF fold assignments are identical,
so `PROBA[a]` and `PROBA[b]` are per-document predictions on the same held-out
samples — the pairing is valid.

In [6]:
CONTRASTS = [
    ("v2",   "v1",   "V2 custom   vs  V1 built-in   (RQ3 primary)"),
    ("v2",   "surf", "V2 custom   vs  surface only"),
    ("v1",   "surf", "V1 built-in vs  surface only"),
]

paired_results = {}
print(f"{'contrast':52s}  {'dAUC':>8s}  {'95% CI':>20s}  {'P(<=0)':>8s}")
for a, b, label in CONTRASTS:
    m, ci, p = paired_bootstrap(PROBA[a], PROBA[b], y, n=2000, seed=SEED)
    flag = "  *" if (ci[0] > 0 or ci[1] < 0) else ""
    print(f"{label:52s}  {m:+.4f}  [{ci[0]:+.4f},{ci[1]:+.4f}]  {p:6.3f}{flag}")
    paired_results[(a, b)] = (m, ci, p)

contrast                                                  dAUC                95% CI    P(<=0)
V2 custom   vs  V1 built-in   (RQ3 primary)           +0.1133  [+0.0314,+0.1944]   0.004  *
V2 custom   vs  surface only                          +0.0626  [+0.0035,+0.1189]   0.014  *
V1 built-in vs  surface only                          -0.0507  [-0.1322,+0.0314]   0.889


## Block 6 — Verdict (RQ3)

All three outcomes are publishable: V2 significantly better, no significant difference,
or V1 better. What matters is stating which one happened and what it implies. A
non-significant result does not mean the prompts are equivalent — it means the sample
is too small to distinguish them at conventional thresholds.

In [7]:
auc_surf = float(table.loc[table.model.str.startswith("Surface"), "AUC"].iloc[0])
auc_v1   = float(table.loc[table.model.str.contains("V1"),        "AUC"].iloc[0])
auc_v2   = float(table.loc[table.model.str.contains("V2 custom"), "AUC"].iloc[0])

d_v2_v1, ci_v2_v1, p_v2_v1 = paired_results[("v2", "v1")]
d_v2_s,  ci_v2_s,  p_v2_s  = paired_results[("v2", "surf")]
d_v1_s,  ci_v1_s,  p_v1_s  = paired_results[("v1", "surf")]

print("=" * 70)
print("RQ3 — Prompt sensitivity: built-in vs custom V2 prompt")
print("=" * 70)
print(f"\nCV AUC:")
print(f"  surface only   : {auc_surf:.3f}")
print(f"  V1 built-in    : {auc_v1:.3f}")
print(f"  V2 custom      : {auc_v2:.3f}")
print()
print(f"Paired bootstrap (2000 resamples):")
print(f"  V2 vs V1  : dAUC {d_v2_v1:+.4f}  CI [{ci_v2_v1[0]:+.4f},{ci_v2_v1[1]:+.4f}]  P(<=0) {p_v2_v1:.3f}")
print(f"  V2 vs surf: dAUC {d_v2_s:+.4f}  CI [{ci_v2_s[0]:+.4f},{ci_v2_s[1]:+.4f}]  P(<=0) {p_v2_s:.3f}")
print(f"  V1 vs surf: dAUC {d_v1_s:+.4f}  CI [{ci_v1_s[0]:+.4f},{ci_v1_s[1]:+.4f}]  P(<=0) {p_v1_s:.3f}")
print()

if ci_v2_v1[0] > 0:
    verdict = (
        f"The custom V2 prompt significantly outperforms the built-in library prompt "
        f"(dAUC {d_v2_v1:+.3f}, CI entirely above zero, P(<=0)={p_v2_v1:.3f}). "
        f"The count + evidence schema extracts more discriminative signal than the "
        f"categorical V1 encoding on this corpus."
    )
elif ci_v2_v1[1] < 0:
    verdict = (
        f"The built-in library prompt outperforms the custom V2 prompt "
        f"(dAUC {d_v2_v1:+.3f}, CI entirely below zero, P(>=0)={1-p_v2_v1:.3f}). "
        f"The V1 categorical encoding retains more signal on this corpus."
    )
else:
    direction = "V2 custom" if d_v2_v1 > 0 else "V1 built-in"
    verdict = (
        f"No statistically significant difference between the two prompt strategies "
        f"(dAUC {d_v2_v1:+.3f}, CI [{ci_v2_v1[0]:+.3f},{ci_v2_v1[1]:+.3f}], "
        f"P(<=0)={p_v2_v1:.3f}). Point estimate favours {direction}. "
        f"Report: the sample is too small to distinguish the two approaches at "
        f"conventional significance thresholds; the CI quantifies the uncertainty."
    )

print("VERDICT:", verdict)
print()
print("TEST SET: not loaded. This notebook is training-only.")

RQ3 — Prompt sensitivity: built-in vs custom V2 prompt

CV AUC:
  surface only   : 0.743
  V1 built-in    : 0.693
  V2 custom      : 0.806

Paired bootstrap (2000 resamples):
  V2 vs V1  : dAUC +0.1133  CI [+0.0314,+0.1944]  P(<=0) 0.004
  V2 vs surf: dAUC +0.0626  CI [+0.0035,+0.1189]  P(<=0) 0.014
  V1 vs surf: dAUC -0.0507  CI [-0.1322,+0.0314]  P(<=0) 0.889

VERDICT: The custom V2 prompt significantly outperforms the built-in library prompt (dAUC +0.113, CI entirely above zero, P(<=0)=0.004). The count + evidence schema extracts more discriminative signal than the categorical V1 encoding on this corpus.

TEST SET: not loaded. This notebook is training-only.
